In [0]:
dbutils.widgets.text("market_table", "markets")

In [0]:
from pyspark.sql.types import StructField, StringType, IntegerType, StructType, DateType, DecimalType


In [0]:
CATALOG = 'dbassociate'
BRONZE_SCHEMA = 'bronze'

In [0]:
markets_table = dbutils.widgets.get("market_table")

In [0]:
VOLUME_PATH = '/Volumes/dbassociate/default/vol_landing/session_09'
DEVOLUCION_CSV = 'devoluciones_retail.csv'
TIENDAS_CSV = 'tiendas_retail.csv'

In [0]:
refunds_schema = StructType([
  StructField("devolucion_id", StringType()),
  StructField("order_id", StringType()),
  StructField("fecha_devolucion", DateType()),
  StructField("cliente_id", StringType()),
  StructField("motivo", StringType()),
  StructField("monto_devuelto", DecimalType(10,2)),
  StructField("estado_devolucion", StringType())
])

markets_schema = StructType([
  StructField("tienda_id", StringType()),
  StructField("nombre", StringType()),
  StructField("ciudad", StringType()),
  StructField("region", StringType()),
  StructField("gerente", StringType()),
  StructField("fecha_apertura", DateType()),
  StructField("capacidad_m2", IntegerType())
])

In [0]:
from pyspark.sql import functions as F

In [0]:
df_market = (
    spark.read
        .option('header', 'true')
        .schema(markets_schema)
        .csv(f'{VOLUME_PATH}/{TIENDAS_CSV}')
        .withColumn('ingestion_at', F.current_timestamp())
        .withColumn('source_file', F.col('_metadata.file_name'))
)

In [0]:
%sql

CREATE TABLE IF NOT EXISTS dbassociate.bronze.markets (
    tienda_id string,
    nombre string,
    ciudad string,
    region string,
    gerente string,
    fecha_apertura date,
    capacidad_m2 int,
    ingestion_at timestamp,
    source_file string
)
USING DELTA

In [0]:
(
    df_market.write
        .format('delta')
        .mode('overwrite')
        .saveAsTable(f'{CATALOG}.{BRONZE_SCHEMA}.{markets_table}')
)